# Merge Datasets
| Source | File | Gives us |
|---|---|---|
| VGChartz | `Video_Games_Sales_as_at_22_Dec_2016.csv` | sales by region, some scores — **base table** |
| Metacritic | `all_games.csv` | critic + user scores, all platforms |
| Steam | `games.csv` | PC games, estimated owners, Steam reviews |

## Setup

In [85]:
import re
import pandas as pd
import unicodedata

RAW = "data/raw/"
CLEAN = "data/clean/"

## Load the files

**Steam file fixes:** the header row in `games.csv` has two column names stuck together (`"DiscountDLC count"` should be `"Discount"` and `"DLC count"`). The data rows have one more column than the header, so pandas moves everything over by one, making `AppID` become the index and every column hold the wrong values.

**Metacritic file fixes:** every platform name starts with a space (`" PC"`, `" PlayStation 4"`).

In [86]:
vg = pd.read_csv(RAW + "Video_Games_Sales_as_at_22_Dec_2016.csv")

mc = pd.read_csv(RAW + "all_games.csv")

steam_cols = pd.read_csv(RAW + "games.csv", nrows=0).columns.tolist()
i = steam_cols.index("DiscountDLC count")
steam_cols[i:i+1] = ["Discount", "DLC count"]
steam = pd.read_csv(
    RAW + "games.csv",
    names=steam_cols, skiprows=1,
    usecols=["AppID", "Name", "Release date", "Estimated owners",
             "Metacritic score", "Positive", "Negative", "Genres", "Publishers", "Developers"],
)

print("VGChartz:  ", vg.shape)
print("Metacritic:", mc.shape)
print("Steam:     ", steam.shape)

VGChartz:   (16719, 16)
Metacritic: (18800, 6)
Steam:      (125855, 10)


## Inspect each file

Before changing anything, get to know each table. For **each** of `vg`, `mc`, `steam`, look at:
- `.head()` and `.columns`
- `.dtypes` — which columns are text that should be numbers?
- `.isna().sum()` — what's missing?
- the platform column's `.value_counts()` (Steam has none — every row is PC)
- the year / date column's range

Write what you notice as a markdown cell under each one. This becomes your Data Description section.

In [87]:
vg.dtypes

Name                   str
Platform               str
Year_of_Release    float64
Genre                  str
Publisher              str
NA_Sales           float64
EU_Sales           float64
JP_Sales           float64
Other_Sales        float64
Global_Sales       float64
Critic_Score       float64
Critic_Count       float64
User_Score             str
User_Count         float64
Developer              str
Rating                 str
dtype: object

In [88]:
vg.isna().sum()

Name                  2
Platform              0
Year_of_Release     269
Genre                 2
Publisher            54
NA_Sales              0
EU_Sales              0
JP_Sales              0
Other_Sales           0
Global_Sales          0
Critic_Score       8582
Critic_Count       8582
User_Score         6704
User_Count         9129
Developer          6623
Rating             6769
dtype: int64

In [89]:
(vg["User_Score"] == "tbd").sum()

np.int64(2425)

In [90]:
display(vg["Platform"].value_counts())
vg["Year_of_Release"].describe()

Platform
PS2     2161
DS      2152
PS3     1331
Wii     1320
X360    1262
PSP     1209
PS      1197
PC       974
XB       824
GBA      822
GC       556
3DS      520
PSV      432
PS4      393
N64      319
XOne     247
SNES     239
SAT      173
WiiU     147
2600     133
NES       98
GB        98
DC        52
GEN       29
NG        12
SCD        6
WS         6
3DO        3
TG16       2
GG         1
PCFX       1
Name: count, dtype: int64

count    16450.000000
mean      2006.487356
std          5.878995
min       1980.000000
25%       2003.000000
50%       2007.000000
75%       2010.000000
max       2020.000000
Name: Year_of_Release, dtype: float64

### VGChartz notes

Most of these columns already have good data types. One thing we do need to change is User_Score. It loads as text because some rows say "tbd", so we need to turn those into missing values and then make the column a float so it matches the other score columns.

A few other type fixes:
- **Year_of_Release** should be a whole number. Sinas ce it has missing values, it hto use pandas' nullable Int64 type instead of a regular int.
- **Critic_Count** and **User_Count** can also be Int64 for the same reason.

There are also quite a few columns with missing data:
- Critic_Score: 8,582 missing
- Critic_Count: 8,582 missing
- User_Score: 6,704 missing, **plus 2,425 more that say "tbd"**, so about 9,129 games have no user score
- User_Count: 9,129 missing
- Developer: 6,623 missing
- Rating (ESRB): 6,769 missing
- Year_of_Release: 269 missing, Publisher: 54, Name and Genre: 2 each

The sales columns aren't missing anything. PC only has 974 rows out of 16,719, which is the main reason we're bringing in the other two datasets.

In [91]:
# mc
mc.dtypes

name              str
platform          str
release_date      str
summary           str
meta_score      int64
user_review       str
dtype: object

In [92]:
mc.isna().sum()

name              0
platform          0
release_date      0
summary         114
meta_score        0
user_review       0
dtype: int64

In [93]:
(mc["user_review"] == "tbd").sum()

np.int64(1365)

In [94]:
display(mc["platform"].value_counts())
mc["meta_score"].describe()

platform
PC                  4864
PlayStation 4       2056
Xbox 360            1644
PlayStation 2       1414
Switch              1399
PlayStation 3       1256
Xbox One            1179
Xbox                 789
DS                   720
Wii                  655
PSP                  512
GameCube             448
Game Boy Advance     438
3DS                  396
PlayStation Vita     257
PlayStation          187
Wii U                184
Dreamcast            125
PlayStation 5        124
Xbox Series X         77
Nintendo 64           71
Stadia                 5
Name: count, dtype: int64

count    18800.000000
mean        70.648883
std         12.225006
min         20.000000
25%         64.000000
50%         72.000000
75%         80.000000
max         99.000000
Name: meta_score, dtype: float64

In [95]:
pd.to_datetime(mc["release_date"]).dt.year.agg(["min", "max"])

min    1995
max    2021
Name: release_date, dtype: int32

### Metacritic notes

This file has 18,800 rows and only 6 columns: name, platform, release_date, summary, meta_score, and user_review.

At first it looks like almost nothing is missing. isna() only shows 114 missing summaries. But **user_review has 1,365 rows that say "tbd"**, which is also why it loads as text instead of a number. Those need to become missing values, and then the column should be a float. It's out of 10, like VG's User_Score.

Other things we need to fix before merging:
- **meta_score** is already a whole number from 20 to 99, which is the same scale as VG's Critic_Score, so it doesn't need converting.
- **platform** values all start with a space (like " PC"), and they're full names ("PlayStation 4") instead of the short codes VG uses ("PS4"). We'll need to strip the spaces and map them to VG's codes.
- **release_date** is text like "November 23, 1998". It all converts to dates fine, and we can pull a year out of it to match VG. Years go from 1995 to 2021.
- **summary** is just a description, and we won't need it for the analysis.

In [96]:
# steam
steam.dtypes

AppID               int64
Name                  str
Release date          str
Estimated owners      str
Metacritic score    int64
Positive            int64
Negative            int64
Developers            str
Publishers            str
Genres                str
dtype: object

In [97]:
steam.isna().sum()

AppID                  0
Name                   1
Release date           0
Estimated owners       0
Metacritic score       0
Positive               0
Negative               0
Developers          8449
Publishers          8922
Genres              8423
dtype: int64

In [98]:
display((steam["Metacritic score"] == 0).sum())
(steam["Estimated owners"] == "0 - 0").sum()

np.int64(121597)

np.int64(24579)

In [99]:
display(steam["Estimated owners"].value_counts())
pd.to_datetime(steam["Release date"], errors="coerce").dt.year.agg(["min", "max"])

Estimated owners
0 - 20000                75705
0 - 0                    24579
20000 - 50000            11397
50000 - 100000            5355
100000 - 200000           3454
200000 - 500000           2856
500000 - 1000000          1155
1000000 - 2000000          729
2000000 - 5000000          405
5000000 - 10000000         125
10000000 - 20000000         51
20000000 - 50000000         31
50000000 - 100000000         9
100000000 - 200000000        4
Name: count, dtype: int64

min    1997
max    2026
Name: Release date, dtype: int32

### Steam notes

This file is big, with 125,855 games, so we only loaded the columns we need.

Like Metacritic, isna() makes it look like the scores are complete, but they're not. Steam uses **0 to mean "no score."**
- **Metacritic score:** 121,597 games have a 0, so only about **4,258 games actually have a score**. We need to change those 0s to missing values.
- **Estimated owners:** 24,579 games say "0 - 0", which basically means no data. This column is text ranges like "0 - 20000", not real numbers, so we'd have to split it into a low and high number if we use it. It's also an estimate of owners, not sales, so it can't be added to VG's sales.

Actually missing values: Developers 8,449, Publishers 8,922, Genres 8,423, and Name 1.

Other things to note:
- Release years go from 1997 to 2026, and 10,150 games came out by 2016. This date is when the game came out *on Steam*, which can be later than its original release.
- There are 1,210 repeated names but no repeated AppIDs, so some different games share the same name. We'll need to watch for that when merging.
- There's no platform column, since everything here is PC.
- **Positive** and **Negative** are Steam review counts, which could work as a user rating since Steam's own user score column is basically empty.

## Clean each table so they can match

Things to fix *before* merging:
- **Metacritic:** strip the spaces from `platform`; turn `"tbd"` in `user_review` into missing and make it numeric; pull a `year` out of `release_date`
- **VGChartz:** same `"tbd"` problem in `User_Score`
- **Steam:** pull a `year` out of `Release date`; `Metacritic score` uses **0** for "no score" — that should be missing, not zero

In [100]:
# remove spaces
mc['platform'] = mc['platform'].str.strip()

In [101]:
# turn tbd into null values and make the user_review column numeric
mc['user_review'] = mc['user_review'].replace('tbd', pd.NA)
mc['user_review'] = pd.to_numeric(mc['user_review'])

In [102]:
# make a year column
mc['year'] = pd.to_datetime(mc['release_date']).dt.year

In [103]:
# turn tbd into null values and make the column numeric
vg['User_Score'] = vg['User_Score'].replace('tbd', pd.NA)
vg['User_Score'] = pd.to_numeric(vg['User_Score'])

In [104]:
# make a year column
steam['year'] = pd.to_datetime(steam['Release date']).dt.year

In [105]:
# turn 0 into null values
steam['Metacritic score'] = steam['Metacritic score'].mask(steam['Metacritic score'] == 0)

## Make the keys

In [106]:
def make_key(name: str):
    if pd.isna(name):
        return pd.NA

    name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    name = name.lower()
    name = re.sub(r"['’]", "", name)
    name = name.replace("&", " and ")
    name = re.sub(r"[^a-z0-9]+", " ", name)
    
    return name.strip()

In [107]:
vg['name_key'] = vg['Name'].apply(make_key)
mc['name_key'] = mc['name'].apply(make_key)
steam['name_key'] = steam['Name'].apply(make_key)

steam['name_key']

0                 black dragon mage playtest
1         supipara chapter 1 spring has come
2          mystery solitaire the black raven
3                            vtuber paranoia
4                              maze quest vr
                         ...                
125850                      sotsugyo omedeto
125851                       island on cloud
125852                          rosie planet
125853             fear effect 2 retro helix
125854                     project liminal 2
Name: name_key, Length: 125855, dtype: str

In [108]:
# make platform names match
platform_map = {
    "PC": "PC",
    "PlayStation": "PS",
    "PlayStation 2": "PS2",
    "PlayStation 3": "PS3",
    "PlayStation 4": "PS4",
    "PlayStation 5": "PS5",
    "PlayStation Vita": "PSV",
    "PSP": "PSP",
    "Xbox": "XB",
    "Xbox 360": "X360",
    "Xbox One": "XOne",
    "Xbox Series X": "XboxSX",
    "Nintendo 64": "N64",
    "GameCube": "GC",
    "Wii": "Wii",
    "Wii U": "WiiU",
    "Switch": "Switch",
    "DS": "DS",
    "3DS": "3DS",
    "Game Boy Advance": "GBA",
    "Dreamcast": "DC",
    "Stadia": "Stadia",
}

mc['platform'] = mc['platform'].replace(platform_map)
mc['platform']

0         N64
1          PS
2         PS3
3          DC
4        X360
         ... 
18795    X360
18796      XB
18797     PS4
18798      PC
18799      PC
Name: platform, Length: 18800, dtype: str

In [109]:
# drop duplicates
mc = mc.drop_duplicates(subset=['name_key','platform'], keep='first')

In [110]:
# drop rows with no name, then duplicate name_key + Platform
vg = vg.dropna(subset=['Name'])
vg = vg.drop_duplicates(subset=['name_key', 'Platform'], keep='first')

## Merge Metacritic onto VGChartz

In [111]:
# keep only the columns we need, and rename so nothing collides with VG's columns
mc_merge = mc.rename(columns={
    'platform': 'Platform',
    'name': 'mc_name',
    'meta_score': 'mc_critic_score',
    'user_review': 'mc_user_score',
    'year': 'mc_year',
})
mc_merge = mc_merge[['name_key', 'Platform', 'mc_name', 'mc_critic_score', 'mc_user_score', 'mc_year']]
mc_merge.head()

,name_key,Platform,mc_name,mc_critic_score,mc_user_score,mc_year
0,the legend of zelda ocarina of time,N64,The Legend of Zelda: Ocarina of Time,99,9.1,1998
1,tony hawks pro skater 2,PS,Tony Hawk's Pro Skater 2,98,7.4,2000
2,grand theft auto iv,PS3,Grand Theft Auto IV,98,7.7,2008
3,soulcalibur,DC,SoulCalibur,98,8.4,1999
4,grand theft auto iv,X360,Grand Theft Auto IV,98,7.9,2008


In [112]:
merged = vg.merge(
    mc_merge,
    on=['name_key', 'Platform'],
    how='outer',
    indicator=True,
    validate='1:1',
)
print(f"VG: {len(vg)}   Metacritic: {len(mc_merge)}   merged: {len(merged)}")
merged['_merge'].value_counts()

VG: 16710   Metacritic: 18686   merged: 28647


_merge
right_only    11937
left_only      9961
both           6749
Name: count, dtype: int64

In [113]:
# VG games that didn't find a Metacritic match
unmatched = merged[merged['_merge'] == 'left_only']
unmatched[['Name', 'Platform', 'Year_of_Release', 'Critic_Score']].sample(20, random_state=1)

,Name,Platform,Year_of_Release,Critic_Score
18107,Power Pro Kun Pocket 8,DS,2005.0,NaN
8937,Garfield Gets Real,DS,2009.0,39.0
15886,NCAA Final Four 2001,PS2,2000.0,49.0
22082,Spy Kids Challenger,GBA,2002.0,NaN
14607,Mobile Suit Gundam: Extreme VS Full Boost,PS3,2014.0,NaN
13778,Mass Effect Trilogy,X360,2012.0,NaN
11271,Jampack Summer 2002,PS2,2002.0,NaN
20852,Shin Sangoku Musou Online: Souten Ranbu,PS3,2010.0,NaN
5637,Digimon World: Next Order,PSV,2016.0,NaN
6361,Dragon Ball Z Super Gokuden: Kakusei-Hen,SNES,1995.0,NaN


In [114]:
both = merged[(merged['_merge'] == 'both') & merged['Critic_Score'].notna()]
(both['Critic_Score'] == both['mc_critic_score']).mean()

np.float64(0.987098804279421)

### Merge results

- 6,749 games matched in both, 9,961 are only in VG, and 11,937 are only in Metacritic, so 28,647 rows total.
- The VG games that didn't match are mostly games Metacritic just doesn't have, like older Japanese games, kids' games, and SNES-era games from before 1995. So it's missing data, not a broken key.
- **The matches are right:** where VG already had a critic score, the Metacritic score is the same 98.7% of the time. That also shows VG's scores originally came from Metacritic, which is why the extra Metacritic *games* matter more than filling in VG's missing scores.

In [115]:
# Metacritic only rows have no VG name or year, so fill them in from Metacritic
merged['Name'] = merged['Name'].fillna(merged['mc_name'])
merged['Year_of_Release'] = merged['Year_of_Release'].fillna(merged['mc_year'])

# keep a readable record of where each row came from
merged = merged.rename(columns={'_merge': 'mc_match'})
merged['mc_match'] = merged['mc_match'].astype(str).replace(
    {'both': 'vg+mc', 'left_only': 'vg only', 'right_only': 'mc only'})
merged = merged.drop(columns=['mc_name'])

merged['mc_match'].value_counts()

mc_match
mc only    11937
vg only     9961
vg+mc       6749
Name: count, dtype: int64

## Add in Steam

**Which Steam games we keep:** only the ones with a Metacritic score. Most of the 125,855 Steam games are tiny and have no scores, and they would swamp every comparison. Keeping games with a score keeps the PC side as "games critics reviewed," same as the console side, and fits our plan of keeping rows that have ratings.

Some of these games share the same `name_key` (68 of them), so we keep the one with the most Steam reviews.

In [116]:
before = len(steam)
steam_f = steam[steam['Metacritic score'].notna()].copy()
print(f"Steam: {before} -> {len(steam_f)} games with a Metacritic score")

# keep the version with the most Steam reviews when names collide
steam_f['total_reviews'] = steam_f['Positive'] + steam_f['Negative']
steam_f = steam_f.sort_values('total_reviews', ascending=False).drop_duplicates('name_key', keep='first')
print(f"After dropping duplicate names: {len(steam_f)}")

Steam: 125855 -> 4258 games with a Metacritic score
After dropping duplicate names: 4190


`Estimated owners` is text like `"200000 - 500000"`, so we split it into a low and high number. These are **owners, not sales**, so they stay in their own columns and never get added to VG's sales.

In [117]:
owners = steam_f['Estimated owners'].str.split(' - ', expand=True).astype(int)
steam_f['steam_owners_low'] = owners[0]
steam_f['steam_owners_high'] = owners[1]

steam_merge = steam_f.rename(columns={
    'AppID': 'steam_appid',
    'Name': 'steam_name',
    'year': 'steam_year',
    'Metacritic score': 'steam_critic_score',
    'Positive': 'steam_positive',
    'Negative': 'steam_negative',
    'Genres': 'steam_genres',
})
steam_merge = steam_merge[['name_key', 'steam_appid', 'steam_name', 'steam_year', 'steam_critic_score',
                           'steam_positive', 'steam_negative', 'steam_owners_low', 'steam_owners_high',
                           'steam_genres']]
steam_merge.head()

,name_key,steam_appid,steam_name,steam_year,steam_critic_score,steam_positive,steam_negative,steam_owners_low,steam_owners_high,steam_genres
4193,dota 2,570,Dota 2,2013,90.0,2037143,461826,100000000,200000000,"Action,Strategy,Free To Play"
59122,grand theft auto v legacy,271590,Grand Theft Auto V Legacy,2015,96.0,1739980,250576,50000000,100000000,"Action,Adventure"
10927,terraria,105600,Terraria,2011,83.0,1373979,35494,20000000,50000000,"Action,Adventure,Indie,RPG"
5377,rust,252490,Rust,2018,69.0,1071135,156649,20000000,50000000,"Action,Adventure,Indie,Massively Multiplayer,RPG"
57377,team fortress 2,440,Team Fortress 2,2007,92.0,1044264,117208,50000000,100000000,"Action,Free To Play"


Steam is all PC, so we only merge it with the PC rows, then stack everything back together.

We match on **name only, not year.** Steam's date is when a game came out *on Steam*, and lots of older games came to Steam years later (like *Anachronox*: 2001, on Steam in 2013). We checked the 238 matches where the years are more than 2 apart, and almost all are the same game with the exact same score. So we keep the original year and only use Steam's year for Steam-only games.

In [118]:
pc = merged[merged['Platform'] == 'PC']
not_pc = merged[merged['Platform'] != 'PC']
print(f"PC rows before Steam: {len(pc)}")

pc = pc.merge(steam_merge, on='name_key', how='outer', indicator=True, validate='1:1')
display(pc['_merge'].value_counts())

# Steam-only rows: fill in platform, name, and year
pc['Platform'] = 'PC'
pc['Name'] = pc['Name'].fillna(pc['steam_name'])
pc['Year_of_Release'] = pc['Year_of_Release'].fillna(pc['steam_year'])
pc['mc_match'] = pc['mc_match'].fillna('steam only')
pc = pc.rename(columns={'_merge': 'steam_match'})
pc['steam_match'] = pc['steam_match'].astype(str).replace(
    {'both': 'matched', 'left_only': 'no steam', 'right_only': 'steam only'})
pc = pc.drop(columns=['steam_name'])

merged = pd.concat([not_pc, pc], ignore_index=True)
print(f"Total rows: {len(merged)}")

PC rows before Steam: 5121


_merge
left_only     3091
right_only    2160
both          2030
Name: count, dtype: int64

Total rows: 30807


In [ ]:
# how well do the Metacritic and Steam scores agree on games that matched?
m = merged[(merged['steam_match'] == 'matched') & merged['mc_critic_score'].notna()]
(m['mc_critic_score'] == m['steam_critic_score']).mean()

np.float64(0.9595)

## Combine overlapping columns and save

The same score can come from more than one source, so we make one `critic_score` and one `user_score` column. Order: **VG first** (our original source), then Metacritic, then Steam. The order barely matters since the sources agree almost every time (98.7% VG vs. Metacritic, about 96% Metacritic vs. Steam). Steam has no usable user score, so `user_score` only comes from VG and Metacritic. All the original score columns stay in the file so you can see where each value came from.

In [120]:
merged['critic_score'] = (merged['Critic_Score']
                          .fillna(merged['mc_critic_score'])
                          .fillna(merged['steam_critic_score']))
merged['user_score'] = merged['User_Score'].fillna(merged['mc_user_score'])

### Last cleanup

- Years and count columns should be whole numbers. They have missing values, so they use pandas' nullable `Int64` type.
- `steam_match` is blank for every non-PC row, since Steam only has PC. We fill it with `"not pc"` so a blank never looks like missing data.
- Put the columns in a sensible order: game info, sales, combined scores, then the source-specific columns.

In [121]:
for col in ['Year_of_Release', 'Critic_Count', 'User_Count', 'mc_year', 'steam_year', 'steam_appid',
            'steam_critic_score', 'steam_positive', 'steam_negative', 'steam_owners_low', 'steam_owners_high']:
    merged[col] = merged[col].astype('Int64')
merged['mc_critic_score'] = merged['mc_critic_score'].astype('Int64')

merged['steam_match'] = merged['steam_match'].fillna('not pc')

merged = merged[[
    # game info
    'Name', 'Platform', 'Year_of_Release', 'Genre', 'Publisher', 'Developer', 'Rating',
    # sales (VG only, millions of units)
    'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales',
    # combined scores
    'critic_score', 'user_score',
    # VG's original scores
    'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count',
    # Metacritic
    'mc_critic_score', 'mc_user_score', 'mc_year',
    # Steam
    'steam_appid', 'steam_year', 'steam_critic_score', 'steam_positive', 'steam_negative',
    'steam_owners_low', 'steam_owners_high', 'steam_genres',
    # where each row came from
    'name_key', 'mc_match', 'steam_match',
]]
merged.dtypes

Name                      str
Platform                  str
Year_of_Release         Int64
Genre                     str
Publisher                 str
Developer                 str
Rating                    str
NA_Sales              float64
EU_Sales              float64
JP_Sales              float64
Other_Sales           float64
Global_Sales          float64
critic_score          float64
user_score            float64
Critic_Score          float64
Critic_Count            Int64
User_Score            float64
User_Count              Int64
mc_critic_score         Int64
mc_user_score         float64
mc_year                 Int64
steam_appid             Int64
steam_year              Int64
steam_critic_score      Int64
steam_positive          Int64
steam_negative          Int64
steam_owners_low        Int64
steam_owners_high       Int64
steam_genres              str
name_key                  str
mc_match                  str
steam_match               str
dtype: object

In [122]:
# Final checks, compared to the original file
orig = pd.read_csv(RAW + "Video_Games_Sales_as_at_22_Dec_2016.csv")
pc_orig = orig['Platform'] == 'PC'
pc_new = merged['Platform'] == 'PC'

print("Rows:", len(orig), "->", len(merged))
print("Duplicate name_key + Platform:", merged.duplicated(['name_key', 'Platform']).sum())
print("PC rows:", pc_orig.sum(), "->", pc_new.sum())
print("PC rows with a critic score:", (pc_orig & orig['Critic_Score'].notna()).sum(), "->",
      (pc_new & merged['critic_score'].notna()).sum())
print("All rows with a critic score:", orig['Critic_Score'].notna().sum(), "->", merged['critic_score'].notna().sum())
print("Rows with sales:", merged['Global_Sales'].notna().sum())

Rows: 16719 -> 30807
Duplicate name_key + Platform: 0
PC rows: 974 -> 7281
PC rows with a critic score: 715 -> 7060
All rows with a critic score: 8137 -> 22631
Rows with sales: 16710


In [124]:
# Rename every column to snake_case. 
merged = merged.rename(columns={
    'Name': 'name',
    'Platform': 'platform',
    'Year_of_Release': 'year',
    'Genre': 'genre',
    'Publisher': 'publisher',
    'Developer': 'developer',
    'Rating': 'esrb_rating',
    'NA_Sales': 'na_sales',
    'EU_Sales': 'eu_sales',
    'JP_Sales': 'jp_sales',
    'Other_Sales': 'other_sales',
    'Global_Sales': 'global_sales',
    'Critic_Score': 'vg_critic_score',
    'Critic_Count': 'vg_critic_count',
    'User_Score': 'vg_user_score',
    'User_Count': 'vg_user_count',
})
merged.columns.tolist()

['name',
 'platform',
 'year',
 'genre',
 'publisher',
 'developer',
 'esrb_rating',
 'na_sales',
 'eu_sales',
 'jp_sales',
 'other_sales',
 'global_sales',
 'critic_score',
 'user_score',
 'vg_critic_score',
 'vg_critic_count',
 'vg_user_score',
 'vg_user_count',
 'mc_critic_score',
 'mc_user_score',
 'mc_year',
 'steam_appid',
 'steam_year',
 'steam_critic_score',
 'steam_positive',
 'steam_negative',
 'steam_owners_low',
 'steam_owners_high',
 'steam_genres',
 'name_key',
 'mc_match',
 'steam_match']

### Final results

- **30,807 rows** (was 16,719), no duplicate games.
- **PC games: 974 -> 7,281**, and PC games with a critic score: **715 -> 7,060**. That was the whole point of the merge.
- Rows with a critic score: 8,137 -> 22,631.
- 16,710 rows still have sales. Every sales question has to use only these rows, and the PC sales undercount (VGChartz only tracks physical copies) is still a limitation.
- About 14,000 rows (the Metacritic-only and Steam-only games) have no VG `Genre`. Metacritic has no genre, and Steam's genres use different names, so genre questions should mostly stick to VG rows.

**How to use this file in the analysis:** filter to the rows each question needs, for example:
```python
sales_df  = df[df['global_sales'].notna()]
critic_df = df[df['critic_score'].notna()]
gap_df    = df.dropna(subset=['critic_score', 'user_score'])
```

In [126]:
merged.to_csv(CLEAN + "games_clean.csv", index=False)

In [128]:
data = pd.read_csv(CLEAN + "games_clean.csv")
data

/var/folders/cq/zj_l9dc55bd9fqfc7310cfv80000gn/T/ipykernel_79096/3138808435.py:1: DtypeWarning: Columns (28: steam_genres) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(CLEAN + "games_clean.csv")


,name,platform,year,genre,publisher,developer,esrb_rating,na_sales,eu_sales,jp_sales,...,steam_year,steam_critic_score,steam_positive,steam_negative,steam_owners_low,steam_owners_high,steam_genres,name_key,mc_match,steam_match
0,0 Day Attack on Earth,X360,2009.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 day attack on earth,mc only,not pc
1,007 Legends,PS2,2012.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,007 legends,mc only,not pc
2,007 Legends,Wii,2012.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,007 legends,mc only,not pc
3,007: Quantum of Solace,DS,2008.0,Action,Activision,Vicarious Visions,T,0.11,0.01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,007 quantum of solace,vg+mc,not pc
4,007: Quantum of Solace,PS2,2008.0,Action,Activision,NaN,NaN,0.17,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,007 quantum of solace,vg+mc,not pc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30802,Zoo Tycoon: Marine Mania,PC,2002.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zoo tycoon marine mania,mc only,no steam
30803,Zool Redimensioned,PC,2021.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zool redimensioned,mc only,no steam
30804,Zuma's Revenge!,PC,2009.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2009.0,78.0,2688.0,140.0,200000.0,500000.0,"Action,Casual",zumas revenge,mc only,matched
30805,Zwei: The Arges Adventure,PC,2018.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2018.0,68.0,379.0,87.0,50000.0,100000.0,"Action,RPG",zwei the arges adventure,mc only,matched
